# Fine-tune Khmer XLS-R on the prepared dataset

Run the cells in order. The smoke-training cell uses only a few examples. Run the full-training cell only after the smoke run completes successfully. The test split is kept out of training.

In [36]:
import sys
import torch

print("Python:", sys.executable)
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Python: /usr/bin/python3
PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [37]:
!pip install -q -r /content/requirements-asr.txt

In [38]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount("/content/drive")

drive_data = Path("/content/drive/MyDrive/khmer_asr/voice_1000")
colab_data = Path("/content/data/voice_1000")
colab_data.parent.mkdir(parents=True, exist_ok=True)
if not colab_data.exists():
    shutil.copytree(drive_data, colab_data)
print("Manifest directory:", colab_data)
print("Files:", sorted(path.name for path in colab_data.iterdir()))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Manifest directory: /content/data/voice_1000
Files: ['baseline_smoke_test.json', 'baseline_test.json', 'manifest_summary.json', 'manifest_validation.json', 'test.csv', 'train.csv', 'validation.csv']


The Colab extension uploads individual files to /content. This cell places the uploaded training script and shared text normalizer in the package folders expected by Python.

In [39]:
from pathlib import Path
import shutil

Path("/content/scripts").mkdir(parents=True, exist_ok=True)
Path("/content/khmer_transliteration").mkdir(parents=True, exist_ok=True)

file_moves = {
    "/content/finetune_asr.py": "/content/scripts/finetune_asr.py",
    "/content/run_asr_baseline.py": "/content/scripts/run_asr_baseline.py",
    "/content/asr_text.py": "/content/khmer_transliteration/asr_text.py",
    "/content/__init__.py": "/content/khmer_transliteration/__init__.py",
}
for source, target in file_moves.items():
    if Path(source).exists():
        shutil.copy2(source, target)
        print("Ready:", target)
    elif not Path(target).exists():
        print("Missing; upload this file:", source)

## 1. Materialize the selected audio

This replays the manifest order on Hugging Face and stores the 793 train, 81 validation, and 126 test WAV files in the temporary Colab runtime. The test files are stored only for later evaluation.

In [40]:
!python /content/scripts/finetune_asr.py --stage materialize --manifest-dir /content/data/voice_1000 --materialized-dir /content/data/voice_1000/materialized_training

Resolving data files: 100% 4447/4447 [00:00<00:00, 24988.44it/s]
Resolving data files: 100% 4447/4447 [00:00<00:00, 26905.49it/s]
Materialized audio to: /content/data/voice_1000/materialized_training
Counts: {'train': 793, 'validation': 81, 'test': 126}


In [41]:
import json
from pathlib import Path

summary_path = Path("/content/data/voice_1000/materialized_training/materialization_summary.json")
print(json.loads(summary_path.read_text(encoding="utf-8")))

{'dataset': 'Digital-Divide-Data/khmer-speech-dataset', 'source_split': 'train', 'seed': 42, 'shuffle_buffer': 5000, 'counts': {'train': 793, 'validation': 81, 'test': 126}, 'target_sample_rate': 16000, 'note': 'Test files are materialized for evaluation but excluded from training.'}


## 2. Smoke training

This runs five optimizer steps on at most 8 training and 4 validation examples. It checks the processor, CTC labels, data collator, GPU, and Trainer before spending time on the full run.

In [45]:
from pathlib import Path
import shutil

Path("/content/scripts").mkdir(parents=True, exist_ok=True)

shutil.copy2(
    "/content/finetune_asr.py",
    "/content/scripts/finetune_asr.py"
)

print("Ready:", Path("/content/scripts/finetune_asr.py").exists())

Ready: True


In [46]:
!python /content/scripts/finetune_asr.py --stage smoke --manifest-dir /content/data/voice_1000 --materialized-dir /content/data/voice_1000/materialized_training --output-dir /content/models/khmer_xlsr_ddd_smoke

Loading weights: 100% 424/424 [00:00<00:00, 4265.49it/s]
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 74, 'bos_token_id': 73}.
{'loss': '5.033', 'grad_norm': '20.38', 'learning_rate': '1e-05', 'epoch': '0.5'}
{'loss': '4.784', 'grad_norm': 'nan', 'learning_rate': '8e-06', 'epoch': '1'}
{'loss': '4.994', 'grad_norm': '14.17', 'learning_rate': '6e-06', 'epoch': '1.5'}
{'loss': '4.386', 'grad_norm': '11.68', 'learning_rate': '4e-06', 'epoch': '2'}
{'loss': '4.142', 'grad_norm': '21.46', 'learning_rate': '2e-06', 'epoch': '2.5'}
100% 5/5 [00:06<00:00,  1.12s/it]
  0% 0/2 [00:00<?, ?it/s]
                                 
{'eval_loss': '1.283', 'eval_cer': '0.3187', 'eval_wer': '4.143', 'eval_sentence_accuracy': '0', 'eval_runtime': '0.4642', 'eval_samples_per_second': '8.616', 'eval_s

In [47]:
import json
from pathlib import Path

smoke_summary = Path("/content/models/khmer_xlsr_ddd_smoke/smoke_training_summary.json")
print(json.loads(smoke_summary.read_text(encoding="utf-8")))

{'stage': 'smoke', 'model': 'vitouphy/wav2vec2-xls-r-300m-khmer', 'train_rows': 8, 'validation_rows': 4, 'train_metrics': {'train_runtime': 7.144, 'train_samples_per_second': 2.8, 'train_steps_per_second': 0.7, 'total_flos': 6896918494224000.0, 'train_loss': 4.667923545837402, 'epoch': 2.5}, 'validation_metrics': {'eval_loss': 1.2833492755889893, 'eval_cer': 0.31875, 'eval_wer': 4.142857142857143, 'eval_sentence_accuracy': 0.0, 'eval_runtime': 0.5891, 'eval_samples_per_second': 6.79, 'eval_steps_per_second': 3.395, 'epoch': 2.5}, 'note': 'The test split was not used during fine-tuning.'}


## 3. Full fine-tuning

Run this only when smoke training succeeds. It uses all train and validation rows for three epochs; the test split remains untouched.

In [48]:
!python /content/scripts/finetune_asr.py --stage full --manifest-dir /content/data/voice_1000 --materialized-dir /content/data/voice_1000/materialized_training --output-dir /content/models/khmer_xlsr_ddd_1000

Loading weights: 100% 424/424 [00:00<00:00, 4384.58it/s]
Skipping unsupported TrainingArguments options: ['group_by_length']
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 74, 'bos_token_id': 73}.
{'loss': '5.431', 'grad_norm': '14.91', 'learning_rate': '1e-05', 'epoch': '0.005038'}
{'loss': '4.764', 'grad_norm': '15.96', 'learning_rate': '9.983e-06', 'epoch': '0.01008'}
{'loss': '4.422', 'grad_norm': '10.05', 'learning_rate': '9.966e-06', 'epoch': '0.01511'}
{'loss': '4.281', 'grad_norm': '9.214', 'learning_rate': '9.95e-06', 'epoch': '0.02015'}
{'loss': '3.851', 'grad_norm': '17.26', 'learning_rate': '9.933e-06', 'epoch': '0.02519'}
{'loss': '4.576', 'grad_norm': '9.286', 'learning_rate': '9.916e-06', 'epoch': '0.03023'}
{'loss': '3.762', 'grad_norm': '8.206', 'learning_rate': '9.

In [49]:
from pathlib import Path
import shutil

local_model = Path("/content/models/khmer_xlsr_ddd_1000")
drive_model = Path("/content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_1000")
drive_model.parent.mkdir(parents=True, exist_ok=True)
shutil.copytree(local_model, drive_model, dirs_exist_ok=True)
print("Saved model to:", drive_model)

Saved model to: /content/drive/MyDrive/khmer_asr/models/khmer_xlsr_ddd_1000


## 4. Evaluate the fine-tuned model on the untouched test split

In [50]:
!python /content/scripts/run_asr_baseline.py --model /content/models/khmer_xlsr_ddd_1000 --manifest-dir /content/data/voice_1000 --split test --max-samples 0 --output /content/data/voice_1000/finetuned_test.json

Resolving data files: 100% 4447/4447 [00:00<00:00, 26945.99it/s]
Resolving data files: 100% 4447/4447 [00:00<00:00, 17097.32it/s]
Loading weights: 100% 424/424 [00:00<00:00, 826.33it/s]
Model: /content/models/khmer_xlsr_ddd_1000
Device: cuda
Successful samples: 126
Errors: 0
Resampled to 16 kHz: 0
Metrics: {'cer': 0.21844481297217616, 'wer': 1.0144578313253012, 'sentence_accuracy': 0.0, 'mean_latency_seconds': 0.09527644272222663, 'median_latency_seconds': 0.08455128950026847, 'p95_latency_seconds': 0.14036579399908078, 'mean_real_time_factor': 0.01390838311227837}
Saved report to: /content/data/voice_1000/finetuned_test.json


In [51]:
from pathlib import Path
import json
import shutil

report_path = Path("/content/data/voice_1000/finetuned_test.json")
report = json.loads(report_path.read_text(encoding="utf-8"))
print("Model:", report["model"])
print("Metrics:", report["metrics"])

drive_report = Path("/content/drive/MyDrive/khmer_asr/voice_1000/finetuned_test.json")
shutil.copy2(report_path, drive_report)
print("Saved report to:", drive_report)

Model: /content/models/khmer_xlsr_ddd_1000
Metrics: {'cer': 0.21844481297217616, 'wer': 1.0144578313253012, 'sentence_accuracy': 0.0, 'mean_latency_seconds': 0.09527644272222663, 'median_latency_seconds': 0.08455128950026847, 'p95_latency_seconds': 0.14036579399908078, 'mean_real_time_factor': 0.01390838311227837}
Saved report to: /content/drive/MyDrive/khmer_asr/voice_1000/finetuned_test.json
